# GeneFlow AI: clasificación taxonómica jerárquica de secuencias de ADN

Este notebook es el cuaderno de trabajo del TFM. Aquí se explora el dataset y, más adelante, se entrenan y evalúan los modelos. **La lógica reutilizable no vive aquí, sino en el paquete `taxonomy_classifier`**, que tiene tests, tipado estricto y un 90 % de cobertura mínima. El notebook solo lo importa y lo usa.

## Requisitos previos

Desde la raíz del repositorio:

| Paso | Comando | Qué hace |
|---|---|---|
| 1 | `uv sync` | Instala el paquete en modo editable junto con los grupos `dev` y `notebooks` |
| 2 | `uv run geneflow prepare` | Descarga las cuatro fuentes (~470 MB), construye el dataset y genera los datos sintéticos (~3 min) |
| 3 | `uv run jupyter lab` | Abre Jupyter. En PyCharm o VS Code, selecciona `.venv` como intérprete |

## De dónde salen los datos

Todas las fuentes contienen el gen del ARN ribosomal de la subunidad pequeña (16S en procariotas, 18S en eucariotas). **GTDB y PR2 son la taxonomía de referencia**: el resto de fuentes se traduce a ellas.

| Fuente | Aporta | Cómo se etiqueta |
|---|---|---|
| **GTDB r232** | 16S de bacterias y arqueas extraídos de genomas completos | Etiqueta propia, hasta especie |
| **PR2 5.1.1** | 18S nuclear de eucariotas, curado a mano | Etiqueta propia, hasta especie. Supergrupo y subdivisión quedan solo en el linaje original |
| **RefSeq 16S** | 16S de cepas tipo | Se traduce a GTDB por especie o, si no existe, por género |
| **SILVA 144 NR99** | Diversidad ambiental de los tres dominios | Se traduce por el nombre más profundo que exista sin ambigüedad en GTDB o PR2. La especie se recupera del nombre de organismo solo si llegó a género, es una especie con nombre formal, existe en GTDB o PR2 y su linaje coincide exactamente |

## Cómo se construye

1. **Lectura y filtrado.** Se descartan las secuencias de menos de 900 pb o de más de 4000 pb, las que tienen más de un 1 % de bases ambiguas, los orgánulos y los registros que no son del gen buscado.
2. **Eliminación de duplicados.** Las secuencias idénticas se fusionan. Si llegan con etiquetas distintas, se quedan con su ancestro común más profundo y se marcan con `label_conflict`. Si ni siquiera coincide el dominio, la secuencia se descarta.
3. **División determinista**, con una semilla fija:

| `split` | Contenido |
|---|---|
| `train` | ~90 % de las secuencias |
| `val`, `test` | ~5 % cada uno, elegidos al azar por secuencia |
| `test_novel_genus` | Todas las secuencias de ~5 % de los géneros, que el modelo nunca ve al entrenar. Mide si sabe generalizar hasta familia ante un género nuevo |

`build_report.json` detalla, para cada fuente, cuántas secuencias se leyeron, cuántas se descartaron y por qué, y hasta qué rango se etiquetaron. También recoge las estadísticas de la fusión y la configuración usada.

## Equilibrado con datos sintéticos

`geneflow augment` genera variantes sintéticas **solo de `train`** para los géneros con pocas secuencias. Validación y test son siempre 100 % reales.

| Regla | Valor |
|---|---|
| Qué se equilibra | Clases de género. Las secuencias sin género se agrupan por su rango más profundo, y cada clase pertenece a un solo reino |
| Suelo | 20 secuencias por clase |
| Copias por secuencia real | Como máximo 5, para que el modelo no memorice una sola secuencia |
| Proporción sintética | **Nunca más sintéticas que reales, dentro de cada reino.** Si un reino se pasa, sus copias se reducen proporcionalmente |
| Cómo se genera cada copia | 0,5 % de sustituciones, 0,1 % de inserciones y 0,1 % de deleciones (~99,4 % de identidad con el original). Es reproducible |
| Colisiones | Se descarta cualquier copia idéntica a una secuencia real de cualquier partición |

El resultado se guarda en `train_synthetic.parquet`, con las columnas `parent_seq_hash` (la secuencia de origen) y `synthetic`. `augment_report.json` recoge cuántas copias hay por reino y cuántas clases siguen por debajo del suelo.

Además, durante el entrenamiento `taxonomy_classifier.training` aplica al vuelo:
- **`SequenceAugmenter`:** recortes aleatorios que simulan lecturas cortas;
- **`balanced_weights`:** pesos de muestreo para el desequilibrio restante.

## Qué hace la celda de configuración

| Variable | Contenido |
|---|---|
| `PROJECT_ROOT` | La raíz del repositorio, detectada buscando `pyproject.toml`. El notebook funciona desde cualquier directorio |
| `LAYOUT` | Las rutas de datos: `LAYOUT.raw_dir(fuente)`, `LAYOUT.interim_dir`, `LAYOUT.processed_dir` |
| `DATASET_PATH`, `REPORT_PATH` | El Parquet final y su informe |
| `AUGMENT_REPORT_PATH` | El informe del equilibrado |
| `SYNTHETIC_PATH` | Las variantes sintéticas de `train` |
| `FIGURES_DIR` | `reports/figures/`: figuras exportadas para la memoria (se crea si no existe) |
| `SOURCES` | Las fuentes de datos, indexadas por su nombre corto (`gtdb_r232`, `pr2_5.1.1`…) |
| `RANK_COLUMNS` | Las columnas taxonómicas en orden jerárquico |
| `SPLITS` | Los nombres de las cuatro particiones |
| `KINGDOMS` | Los seis reinos clásicos |

Además configura el `logging`, cómo muestra Polars las tablas y el estilo de Matplotlib (figuras a 300 ppp). Si el dataset o los datos sintéticos no existen, la celda se detiene con el comando que hay que ejecutar. No carga ningún dato: de eso se encarga la celda de carga.

**Salida.** Muestra dos tablas para comprobar de un vistazo que todo está en su sitio:
- **Los artefactos generados:** dataset, datos sintéticos e informes, con su tamaño y fecha de modificación.
- **Las fuentes:** si están descargadas, cuánto ocupan y si su hash está fijado (RefSeq no lo tiene, porque NCBI lo regenera a diario).

Además define la paleta de las gráficas (`SERIES`, `SURFACE`, `INK`…). Es la paleta de referencia validada para daltonismo, y los colores se asignan siempre en el mismo orden.

## Estructura del dataset

| Columna | Tipo | Descripción |
|---|---|---|
| `seq_hash` | binario | Huella BLAKE2b de 16 bytes de la secuencia. Identifica cada fila |
| `sequence` | texto | Secuencia de ADN normalizada (`U` pasa a `T`, en mayúsculas) |
| `length`, `n_ambiguous` | entero | Longitud y número de bases distintas de `A`, `C`, `G` y `T` |
| `domain` … `species` | texto o nulo | Un rango por columna. Nulo si se desconoce, es de relleno o hubo conflicto entre fuentes |
| `kingdom` | texto o nulo | Reino clásico: `Bacteria`, `Archaea`, `Animalia`, `Fungi`, `Plantae` o `Protista`. Nulo si no se puede determinar o las fuentes discrepan |
| `sources` | lista de texto | Fuentes que aportaron la secuencia (`gtdb`, `pr2`, `refseq`, `silva`) |
| `accessions` | lista de texto | Identificadores originales, como `fuente:accesión` |
| `n_records` | entero | Cuántos registros se fusionaron en esta fila |
| `label_conflict` | booleano | Las fuentes discrepaban y la etiqueta se recortó a su ancestro común |
| `split` | texto | `train`, `val`, `test` o `test_novel_genus` |

**Cómo se asigna `kingdom`:**
- **GTDB y RefSeq** usan el dominio.
- **PR2** usa su linaje original: la subdivisión Metazoa es `Animalia`, la subdivisión Fungi es `Fungi`, las divisiones Streptophyta, Chlorophyta, Prasinodermophyta, Rhodophyta y Glaucophyta son `Plantae` (Archaeplastida fotosintéticas, algas rojas incluidas) y el resto es `Protista`.
- **SILVA**: en procariotas usa el dominio. En eucariotas hereda el reino de su traducción a PR2, solo si no es ambiguo.

`Protista` no es un grupo natural: reúne todos los eucariotas que no son animales, plantas ni hongos.

En eucariotas, las columnas siguen la jerarquía de PR2. Por ejemplo, en un hongo `phylum` = Opisthokonta y `class` = Ascomycota. Es coherente dentro de PR2, pero no coincide con la nomenclatura clásica.


In [3]:
from datetime import datetime
import json
import logging
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import polars as pl

from taxonomy_classifier.data.build import DataLayout
from taxonomy_classifier.data.columns import RANK_COLUMNS
from taxonomy_classifier.data.kingdom import Kingdom
from taxonomy_classifier.data.sources.registry import DEFAULT_SOURCES
from taxonomy_classifier.data.split import Split
from taxonomy_classifier.training.oversampling import training_frame

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").exists()
)

LAYOUT = DataLayout(root=PROJECT_ROOT / "data")
DATASET_PATH = LAYOUT.dataset_path
REPORT_PATH = LAYOUT.report_path
AUGMENT_REPORT_PATH = LAYOUT.augment_report_path
SYNTHETIC_PATH = LAYOUT.synthetic_path
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"

SOURCES = {source.slug: source for source in DEFAULT_SOURCES}
SPLITS = [split.value for split in Split]
KINGDOMS = [kingdom.value for kingdom in Kingdom]

if not DATASET_PATH.exists() or not SYNTHETIC_PATH.exists():
    msg = f"{DATASET_PATH} not found; run 'uv run geneflow prepare' from {PROJECT_ROOT}"
    raise FileNotFoundError(msg)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(name)s: %(message)s",
)

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(80)
pl.Config.set_thousands_separator(".")
pl.Config.set_decimal_separator(",")

SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_SECONDARY = "#52514e"
INK_MUTED = "#898781"
GRID = "#e1e0d9"
BASELINE = "#c3c2b7"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]

plt.rcParams.update(
    {
        "figure.figsize": (10, 5),
        "figure.dpi": 110,
        "figure.facecolor": SURFACE,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "savefig.facecolor": SURFACE,
        "font.family": ["Segoe UI", "DejaVu Sans", "sans-serif"],
        "text.color": INK,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": BASELINE,
        "axes.labelcolor": INK_SECONDARY,
        "axes.titlelocation": "left",
        "axes.titlesize": 12,
        "axes.titleweight": "bold",
        "axes.titlepad": 12,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.axisbelow": True,
        "axes.prop_cycle": plt.cycler(color=SERIES),
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "xtick.color": INK_MUTED,
        "ytick.color": INK_MUTED,
        "xtick.labelcolor": INK_SECONDARY,
        "ytick.labelcolor": INK_SECONDARY,
        "legend.frameon": False,
    }
)


def size(file: Path) -> str:
    size_bytes = file.stat().st_size

    if size_bytes >= 1e6:
        return f"{size_bytes / 1e6:.1f} MB".replace(".", ",")

    return f"{size_bytes / 1e3:.1f} kB".replace(".", ",")


artifacts = pl.DataFrame(
    [
        {
            "artefacto": name,
            "ruta": str(file.relative_to(PROJECT_ROOT)),
            "tamaño": size(file),
            "modificado": datetime.fromtimestamp(file.stat().st_mtime).strftime("%Y-%m-%d %H:%M"),
        }
        for name, file in {
            "dataset": DATASET_PATH,
            "sintéticos": SYNTHETIC_PATH,
            "informe build": REPORT_PATH,
            "informe augment": AUGMENT_REPORT_PATH,
        }.items()
    ]
)

sources = pl.DataFrame(
    [
        {
            "fuente": slug,
            "archivos": len(source.files),
            "descargado": all((LAYOUT.raw_dir(source) / remote.filename).exists() for remote in source.files),
            "tamaño": ", ".join(size(LAYOUT.raw_dir(source) / remote.filename) for remote in source.files),
            "hash fijado": all(remote.pinned for remote in source.files),
        }
        for slug, source in SOURCES.items()
    ]
)

display(artifacts, sources)

## Carga de datos

Esta celda abre los datos, comprueba que están al día y muestra un resumen. **No carga las secuencias en memoria:** todo son `LazyFrame` de Polars, que solo leen del disco lo necesario cuando se llama a `.collect()`.

| Variable | Contenido |
|---|---|
| `dataset` | Todas las secuencias reales, de las cuatro particiones |
| `synthetic` | Las variantes sintéticas de `train`, con `parent_seq_hash` (la secuencia de origen) y `synthetic` |
| `splits` | Diccionario con cada partición real: `splits["val"]`, `splits["test"]`, `splits["test_novel_genus"]`… |
| `train` | `train` real más sintético, que es lo que verá el modelo. La columna `synthetic` distingue unas de otras |
| `build_report`, `augment_report` | Los informes de construcción y de equilibrado, como diccionarios |
| `overview` | Tabla resumen por partición: secuencias, cuántas son sintéticas y qué porcentaje tiene género y especie |

**Salida.** La celda muestra `overview` y una figura con tres paneles, que también se guarda en `reports/figures/dataset_overview.png`:
1. **Secuencias por partición**, separando reales y sintéticas;
2. **`train` por reino**, con el porcentaje de secuencias sintéticas de cada uno. Ningún reino puede pasar del 50 %, porque nunca hay más sintéticas que reales;
3. **Porcentaje de secuencias reales etiquetadas en cada rango**, de dominio a especie.

**Comprobaciones.** La celda se detiene si encuentra una de estas dos situaciones:
- **Los datos no coinciden con sus informes:** el número de secuencias reales o sintéticas del disco no es el que registraron `build` y `augment`.
- **Los datos sintéticos son anteriores al dataset:** se reconstruyó el dataset pero no se volvió a ejecutar `augment`, así que las copias podrían heredar etiquetas antiguas.

En los dos casos, la solución es `uv run geneflow prepare`.


In [4]:
dataset = pl.scan_parquet(DATASET_PATH)
synthetic = pl.scan_parquet(SYNTHETIC_PATH)

build_report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
augment_report = json.loads(AUGMENT_REPORT_PATH.read_text(encoding="utf-8"))

if SYNTHETIC_PATH.stat().st_mtime < DATASET_PATH.stat().st_mtime:
    msg = "Synthetic data is older than the dataset; run 'uv run geneflow augment'"
    raise RuntimeError(msg)

expected = {"real": build_report["merge"]["kept"], "synthetic": augment_report["synthetic"]}
found = {
    "real": dataset.select(pl.len()).collect().item(),
    "synthetic": synthetic.select(pl.len()).collect().item(),
}

if found != expected:
    msg = f"Data on disk {found} does not match its reports {expected}; run 'uv run geneflow prepare'"
    raise RuntimeError(msg)

splits = {split: dataset.filter(pl.col("split") == split) for split in SPLITS}
train = training_frame(DATASET_PATH, SYNTHETIC_PATH)

records = pl.concat(
    [
        dataset.with_columns(pl.lit(value=False).alias("synthetic")),
        synthetic.drop("parent_seq_hash"),
    ],
    how="vertical",
)

overview = (
    records.group_by("split")
    .agg(
        pl.len().alias("sequences"),
        pl.col("synthetic").sum().alias("synthetic"),
        pl.col("genus").is_not_null().mean().mul(100).round(1).alias("% genus"),
        pl.col("species").is_not_null().mean().mul(100).round(1).alias("% species"),
    )
    .with_columns(pl.col("split").cast(pl.Enum(SPLITS)))
    .sort("split")
    .collect()
)

display(overview)


def thousands(value: int) -> str:
    return f"{value:,}".replace(",", ".")


def stacked_bars(axis: plt.Axes, frame: pl.DataFrame, label: str, title: str) -> None:
    labels = frame.get_column(label).to_list()
    real = frame.get_column("real").to_list()
    fake = frame.get_column("synthetic").to_list()

    axis.barh(labels, real, height=0.6, color=SERIES[0], edgecolor=SURFACE, linewidth=2, label="Reales")
    axis.barh(labels, fake, left=real, height=0.6, color=SERIES[1], edgecolor=SURFACE, linewidth=2, label="Sintéticas")

    for position, (count, extra) in enumerate(zip(real, fake, strict=True)):
        note = f"  {thousands(count + extra)}"

        if extra:
            note += f"  ({extra / (count + extra):.0%} sintético)".replace("%", " %")

        axis.text(count + extra, position, note, va="center", fontsize=9, color=INK_SECONDARY)

    axis.set_title(title)
    axis.grid(axis="y", visible=False)
    axis.xaxis.set_major_formatter(lambda value, _: f"{value / 1e6:g} M".replace(".", ","))
    axis.set_xlim(0, max(r + s for r, s in zip(real, fake, strict=True)) * 1.45)


split_counts = (
    overview.select(pl.col("split").cast(pl.String), (pl.col("sequences") - pl.col("synthetic")).alias("real"), "synthetic")
    .reverse()
)

kingdom_counts = (
    train.group_by(pl.col("kingdom").fill_null("sin reino"))
    .agg((~pl.col("synthetic")).sum().alias("real"), pl.col("synthetic").sum().alias("synthetic"))
    .sort(pl.col("real") + pl.col("synthetic"))
    .collect()
)

coverage = dataset.select(pl.col(RANK_COLUMNS).is_not_null().mean().mul(100)).collect().row(0)

figure, (splits_axis, kingdoms_axis, coverage_axis) = plt.subplots(1, 3, figsize=(17, 5), layout="constrained")

stacked_bars(splits_axis, split_counts, "split", "Secuencias por partición")
stacked_bars(kingdoms_axis, kingdom_counts, "kingdom", "Train por reino")

coverage_axis.barh(RANK_COLUMNS[::-1], coverage[::-1], height=0.6, color=SERIES[0], edgecolor=SURFACE, linewidth=2)

for position, value in enumerate(coverage[::-1]):
    coverage_axis.text(value, position, f"  {value:.1f} %".replace(".", ","), va="center", fontsize=9, color=INK_SECONDARY)

coverage_axis.set_title("Secuencias reales etiquetadas por rango")
coverage_axis.set_xlim(0, 118)
coverage_axis.xaxis.set_major_formatter(lambda value, _: f"{value:.0f} %")
coverage_axis.grid(axis="y", visible=False)

figure.legend(*splits_axis.get_legend_handles_labels(), loc="outside upper right", ncols=2)

figure.savefig(FIGURES_DIR / "dataset_overview.png")
plt.show()